In [1]:
import requests
import json

def launch_binder_and_get_jupyter_url(binder_url, connect_timeout=10):
    """
    Launch a BinderHub build using requests with streaming enabled.

    Returns:
        (True, jupyter_url_with_token) on success
        (False, error_message) on failure

    Notes:
    - This function blocks until BinderHub reports either success or failure.
    - No exceptions are raised; all errors are converted to return values.
    """

    # Step 1: initiate the HTTP request (may block until headers are received)
    try:
        resp = requests.get(
            binder_url,
            stream=True,
            headers={"Accept": "text/event-stream"},
            timeout=(connect_timeout, None),  # timeout only for connection, not for streaming
        )
    except Exception as e:
        return False, f"Failed to connect to BinderHub: {e}"

    # Step 2: validate HTTP response
    if resp.status_code != 200:
        return False, f"BinderHub returned HTTP {resp.status_code}"

    # Step 3: read Server-Sent Events (SSE) line by line
    try:
        for raw_line in resp.iter_lines(decode_unicode=True):
            if not raw_line:
                continue

            # BinderHub sends events in SSE format: "data: {...}"
            if not raw_line.startswith("data:"):
                continue

            data = raw_line[5:].strip()

            # Parse JSON payload
            try:
                event = json.loads(data)
            except json.JSONDecodeError:
                continue

            phase = event.get("phase")

            # ---- Optional: real-time logging ----
            print("EVENT:", event, flush=True)

            # Build failed
            if phase == "failed":
                message = event.get("message") or "Binder build failed"
                return False, message

            # Build succeeded and Jupyter server is ready
            if phase == "ready":
                url = event.get("url")
                token = event.get("token")

                if not url:
                    return False, "Binder reported ready phase without a URL"

                # Some BinderHub versions include the token separately,
                # others already embed it in the URL
                # if token and "token=" not in url:
                #     separator = "&" if "?" in url else "?"
                #     url = f"{url}{separator}token={token}"

                return True, {"url": url, "token": token}

        # Stream ended without receiving a final "ready" or "failed" event
        return False, "Binder event stream ended without a final result"

    except Exception as e:
        return False, f"Error while reading Binder event stream: {e}"


In [5]:
binder_base_url = "https://binder.intel4coro.de/build/gh"
repo_path = "yxzhan/cram-vrb-lab/dev"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url)

if ok:
    print("JupyterLab URL:", result)
else:
    print("Binder failed:", result)

EVENT: {'phase': 'waiting', 'message': 'Waiting for build to start...\n'}
EVENT: {'message': 'Picked Git content provider.\n'}
EVENT: {'message': "Cloning into '/tmp/repo2dockernb88rc8x'...\n", 'phase': 'fetching'}
EVENT: {'message': 'Updating files:   9% (157/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  10% (171/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  11% (189/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  12% (206/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  13% (223/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  14% (240/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  15% (257/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  16% (274/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  17% (291/1710)\r', 'phase': 'fetching'}
EVENT: {'message': 'Updating files:  18% (308/1710)\r', 'phase': 'fetching'}
EVENT: {'messag

In [ ]:
binder_base_url = "https://binder.dev.intel4coro.de/build/gh"
repo_path = "yxzhan/cram-vrb-lab/dev"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url)

if ok:
    print("JupyterLab URL:", result['url'] + 'files/demos/web_ui/index.html??token=' + result['token'])
else:
    print("Binder failed:", result)

EVENT: {'phase': 'built', 'imageName': 'intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:53f8660ee7275964c85de89b2e059963bdf79990', 'message': 'Found built image, launching...\n'}
EVENT: {'phase': 'launching', 'message': 'Launching server...\n'}
EVENT: {'phase': 'launching', 'message': 'Server requested\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-28T15:16:01Z [Normal] Successfully assigned binder/jupyter-yxzhan-cram-vrb-lab-4sz34v1x to gpu-worker\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-28T15:16:02Z [Normal] Pulling image "intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:53f8660ee7275964c85de89b2e059963bdf79990"\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-28T15:16:05Z [Normal] Successfully pulled image "intel4coro/yxzhan-2dcram-2dvrb-2dlab-eb909a:53f8660ee7275964c85de89b2e059963bdf79990" in 3.088s (3.088s including waiting). Image size: 16421036462 bytes.\n'}
EVENT: {'phase': 'launching', 'message': '2026-08-28T15:16:05Z [Normal] Created container: notebook\n'}
E